In [14]:

pip install cloudscraper


   ---------------------------------------- 2/2 [cloudscraper]

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import io

url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_squads"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
response = requests.get(url, headers=headers)
html = response.text
soup = BeautifulSoup(html, 'html.parser')

all_squads = []

for table in soup.find_all('table', class_='sortable'):
    header = table.find_previous(['h3', 'h2'])
    team_name = header.text.replace('[edit]', '').strip() if header else 'Unknown'
    
    try:
        df = pd.read_html(io.StringIO(str(table)))[0]
    except Exception:
        continue
        
    if 'Player' in df.columns and 'Club' in df.columns:
        df['Team'] = team_name
        all_squads.append(df)

if all_squads:
    full_squads_df = pd.concat(all_squads, ignore_index=True)
    
    # Drop rows that have missing values in more than 3 fields
    min_non_na = full_squads_df.shape[1] - 3
    full_squads_df.dropna(thresh=min_non_na, inplace=True)
    full_squads_df.reset_index(drop=True, inplace=True)
    
    print(f"Loaded {len(full_squads_df)} players across {len(all_squads)} teams.")
else:
    print("No squads found.")

    

Loaded 1246 players across 48 teams.


### Save DataFrame to CSV File

In [3]:
# Process captain designation and clean player names
import re

# Create Is_Captain column (1 if captain, 0 otherwise)
full_squads_df['Is_Captain'] = full_squads_df['Player'].str.contains('(captain)', case=False, na=False).astype(int)

# Remove (captain) from player names
full_squads_df['Player'] = full_squads_df['Player'].str.replace(r'\s*\(captain\)\s*', '', flags=re.IGNORECASE, regex=True)

# Save updated CSV
storage_path = "world_cup_squadsNew.csv"
full_squads_df.to_csv(storage_path, index=False)

print(f"✓ Updated {full_squads_df['Is_Captain'].sum()} captains identified")
print(f"✓ Player names cleaned and Is_Captain column added")
print(f"✓ File saved to {storage_path}")
print(f"\nFirst few rows:")
print(full_squads_df[['Player', 'Club', 'Team', 'Is_Captain']].head(10))


✓ Updated 48 captains identified
✓ Player names cleaned and Is_Captain column added
✓ File saved to world_cup_squadsNew.csv

First few rows:
             Player                     Club            Team  Is_Captain
0       Matěj Kovář            PSV Eindhoven  Czech Republic           0
1        David Zima            Slavia Prague  Czech Republic           0
2       Tomáš Holeš            Slavia Prague  Czech Republic           0
3      Robin Hranáč           TSG Hoffenheim  Czech Republic           0
4   Vladimír Coufal           TSG Hoffenheim  Czech Republic           0
5  Štěpán Chaloupek            Slavia Prague  Czech Republic           0
6   Ladislav Krejčí  Wolverhampton Wanderers  Czech Republic           1
7   Vladimír Darida           Hradec Králové  Czech Republic           0
8       Adam Hložek           TSG Hoffenheim  Czech Republic           0
9     Patrik Schick         Bayer Leverkusen  Czech Republic           0


C:\Users\sraul\AppData\Local\Temp\ipykernel_23348\191987493.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  full_squads_df['Is_Captain'] = full_squads_df['Player'].str.contains('(captain)', case=False, na=False).astype(int)


In [6]:
group_mapping = {
    'Mexico': 'A', 'South Africa': 'A', 'South Korea': 'A', 'Czech Republic': 'A',
    'Canada': 'B', 'Bosnia and Herzegovina': 'B', 'Qatar': 'B', 'Switzerland': 'B',
    'Brazil': 'C', 'Morocco': 'C', 'Haiti': 'C', 'Scotland': 'C',
    'United States': 'D', 'Paraguay': 'D', 'Australia': 'D', 'Turkey': 'D',
    'Germany': 'E', 'Cura\u00e7ao': 'E', 'Ivory Coast': 'E', 'Ecuador': 'E',
    'Netherlands': 'F', 'Japan': 'F', 'Sweden': 'F', 'Tunisia': 'F',
    'Belgium': 'G', 'Egypt': 'G', 'Iran': 'G', 'New Zealand': 'G',
    'Spain': 'H', 'Cape Verde': 'H', 'Saudi Arabia': 'H', 'Uruguay': 'H',
    'France': 'I', 'Senegal': 'I', 'Iraq': 'I', 'Norway': 'I',
    'Argentina': 'J', 'Algeria': 'J', 'Austria': 'J', 'Jordan': 'J',
    'Portugal': 'K', 'DR Congo': 'K', 'Uzbekistan': 'K', 'Colombia': 'K',
    'England': 'L', 'Croatia': 'L', 'Ghana': 'L', 'Panama': 'L'
}

full_squads_df['Group'] = full_squads_df['Team'].map(group_mapping)

full_squads_df.to_csv(storage_path, index=False)

print(f"\u2713 Group column added and CSV updated")
print(f"\nSample:")
print(full_squads_df[['Player', 'Team', 'Group']].head(10))

✓ Group column added and CSV updated

Sample:
             Player            Team Group
0       Matěj Kovář  Czech Republic     A
1        David Zima  Czech Republic     A
2       Tomáš Holeš  Czech Republic     A
3      Robin Hranáč  Czech Republic     A
4   Vladimír Coufal  Czech Republic     A
5  Štěpán Chaloupek  Czech Republic     A
6   Ladislav Krejčí  Czech Republic     A
7   Vladimír Darida  Czech Republic     A
8       Adam Hložek  Czech Republic     A
9     Patrik Schick  Czech Republic     A


In [5]:
import pandas as pd

codes_extra = pd.read_csv('codes extra.csv')
ids = pd.read_csv('sofascore_player_idsNew.csv')

merged = codes_extra.merge(
    ids,
    left_on='Player Name',
    right_on='Original_Name',
    how='left'
)

merged.to_csv('squads_with_sofascore_idsNew.csv', index=False)

matched = merged['Sofascore_ID'].notna().sum()
total = len(merged)
print(f"\u2713 Saved squads_with_sofascore_idsNew.csv")
print(f"  Total: {total}, Matched with Sofascore ID: {matched}/{total}")
print(f"\nSample:")
cols = ['Player', 'Pos.', 'Team', 'Club_squad', 'Sofascore_ID', 'Sofascore_Name', 'Match_Score']

✓ Saved squads_with_sofascore_idsNew.csv
  Total: 38, Matched with Sofascore ID: 32/38

Sample:
